# Omega free-compute probe (Colab)
Short, finite job (~2 min). Records the machine, runs Omega's seed birth (EMULATED under qemu-aarch64 on x86-64; omc has no x86-64 target), and probes whether the GPU's OpenCL driver accepts SPIR-V. Results print as `OMEGA-FC` lines plus one sha256 receipt. Scratch branch `freecompute/runner-probe` of aien-dev/omega-mv; never merged.

In [ ]:
%%bash
set +e
mkdir -p /content/fc && cd /content/fc
{ echo "OMEGA-FC host=colab date=$(date -u +%FT%TZ)"
  nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv,noheader 2>&1 | sed 's/^/OMEGA-FC gpu=/'
  lscpu | grep -E 'Model name|^CPU\(s\)|Architecture|Vendor ID' | sed 's/^/OMEGA-FC cpu: /'
  uname -srm | sed 's/^/OMEGA-FC kernel=/'
  grep MemTotal /proc/meminfo | sed 's/^/OMEGA-FC /'; } | tee machine.txt

In [ ]:
%%bash
set +e
cd /content/fc
SRC=369dbc96daa1684db2d7c2c7085bbf16b3fd77ee
PIN=2c82480596bbdc50cda9626866209ef662d5834f278f839ff4ce0250c495029f
rm -rf omega && git init -q omega && cd omega && git remote add origin https://github.com/aien-dev/omega-mv.git
git fetch -q --depth 1 origin $SRC && git checkout -q FETCH_HEAD
h=$(sha256sum seed/out/omc | cut -d' ' -f1)
echo "OMEGA-FC source=$(git rev-parse HEAD) omc_sha256=$h pin_ok=$([ $h = $PIN ] && echo yes || echo NO)" | tee -a ../machine.txt
echo "OMEGA-FC native_target=NOT_RUN reason=omc emits AArch64 Linux ELF; x86-64 backend absent (OM_COMPLETION FR-9)" | tee -a ../machine.txt
apt-get -qq install -y qemu-user-static binfmt-support > /dev/null 2>&1
update-binfmts --enable qemu-aarch64 > /dev/null 2>&1
start=$(date +%s); sh seed/omega-from-seed.sh > ../seed-birth.log 2>&1; rc=$?
echo "OMEGA-FC EMULATED seed_birth rc=$rc pass_lines=$(grep -c '^PASS' ../seed-birth.log) seconds=$(( $(date +%s)-start )) omc_after=$(sha256sum seed/out/omc | cut -c1-16)" | tee -a ../machine.txt
tail -3 ../seed-birth.log

In [ ]:
%%bash
set +e
cd /content/fc
apt-get -qq install -y clinfo ocl-icd-libopencl1 > /dev/null 2>&1
ls /etc/OpenCL/vendors/ 2>&1 | sed 's/^/OMEGA-FC icd: /' | tee -a machine.txt
clinfo 2>&1 | grep -E 'Platform Name|Device Name|IL version|Device Version|Driver Version' | sed 's/^ */OMEGA-FC opencl: /' | tee -a machine.txt
clinfo 2>&1 | grep -q 'IL version.*SPIR-V' && echo 'OMEGA-FC opencl_spirv_il=present' || echo 'OMEGA-FC opencl_spirv_il=absent_or_no_platform' | tee -a machine.txt

In [ ]:
%%bash
cd /content/fc
echo "OMEGA-FC receipt_sha256=$(sha256sum machine.txt | cut -d' ' -f1)"
echo '----- paste everything between the lines -----'; cat machine.txt; echo '----- end -----'